# E17 - Playground LoRA y ADR para chatbot financiero


## Imagen guia del tema <!-- data-aem4-visual-assets -->

![e17 adr lora decision](assets/visuals/e17_adr_lora_decision.svg)

Usa esta imagen como punto de partida antes de ejecutar las celdas. La idea es que el alumno vea el flujo completo antes de mirar numeros.


## Para que sirve

Este notebook integra la clase en una decision concreta. Permite cambiar restricciones del caso y comparar alternativas: modelo grande con full fine-tuning, modelo moderado con LoRA o modelo chico sin adaptacion.

**Cuando mostrarlo:** despues de `E11_lora_y_peft_adaptar_sin_reentrenar_todo.ipynb` y como apoyo de `E12_chatbot_financiero_decisiones_de_arquitectura.ipynb`.

**Idea docente:** una decision de arquitectura se justifica con restricciones, no con gustos.


In [ ]:
# Editar restricciones en clase.
caso = {
    "latencia_objetivo_ms": 600,
    "gpu_limitada": True,
    "terminos_nuevos_por_mes": 20,
    "clientes": 8,
    "documentos_largos": True,
    "dominio": "financiero/regulatorio",
}


In [ ]:
alternativas = [
    {
        "nombre": "modelo grande + full fine-tuning",
        "calidad": 5,
        "latencia": 1,
        "costo": 1,
        "adaptacion": 2,
        "storage_base_gb": 14,
        "storage_por_cliente_gb": 14,
    },
    {
        "nombre": "modelo moderado + LoRA",
        "calidad": 4,
        "latencia": 4,
        "costo": 4,
        "adaptacion": 5,
        "storage_base_gb": 14,
        "storage_por_cliente_gb": 0.25,
    },
    {
        "nombre": "modelo chico sin adaptacion",
        "calidad": 2,
        "latencia": 5,
        "costo": 5,
        "adaptacion": 1,
        "storage_base_gb": 4,
        "storage_por_cliente_gb": 0,
    },
]

def score(alt, caso):
    total = alt["calidad"] + alt["latencia"] + alt["costo"] + alt["adaptacion"]
    if caso["gpu_limitada"] and alt["costo"] <= 2:
        total -= 2
    if caso["terminos_nuevos_por_mes"] > 10 and alt["adaptacion"] < 4:
        total -= 2
    if caso["latencia_objetivo_ms"] <= 600 and alt["latencia"] < 3:
        total -= 2
    return total

print("| alternativa | score | storage total | comentario |")
print("|---|---:|---:|---|")
for alt in alternativas:
    total_storage = alt["storage_base_gb"] + caso["clientes"] * alt["storage_por_cliente_gb"]
    comentario = "candidato"
    if alt["latencia"] < 3:
        comentario = "riesgo de latencia"
    if alt["adaptacion"] < 3 and caso["terminos_nuevos_por_mes"] > 10:
        comentario = "riesgo de dominio"
    print(f"| {alt['nombre']} | {score(alt, caso)} | {total_storage:.1f} GB | {comentario} |")


In [ ]:
# Generador de ADR simple para completar en clase.
mejor = max(alternativas, key=lambda alt: score(alt, caso))

print("ADR - Decision propuesta")
print("========================")
print("Contexto:")
print(f"- Dominio: {caso['dominio']}")
print(f"- SLA: {caso['latencia_objetivo_ms']} ms")
print(f"- Clientes: {caso['clientes']}")
print(f"- Terminos nuevos por mes: {caso['terminos_nuevos_por_mes']}")
print()
print("Decision:")
print(f"- Usar: {mejor['nombre']}")
print()
print("Justificacion:")
print("- Balancea calidad, costo, latencia y velocidad de adaptacion.")
print("- Permite explicar el trade-off con numeros visibles.")
print()
print("Riesgos a medir:")
print("- Latencia real con datos largos.")
print("- Calidad en terminos regulatorios nuevos.")
print("- Fragmentacion de tokenizacion en jerga financiera.")


## Preguntas para hacer en clase

1. Que restriccion pesa mas: latencia, costo, calidad o adaptacion
2. Por que LoRA aparece como buena opcion si hay muchos clientes
3. Que riesgo queda aunque elijamos la mejor alternativa
4. Que datos reales pedirian antes de cerrar la decision

## Mini desafio

Cambiar `clientes`, `latencia_objetivo_ms` y `terminos_nuevos_por_mes`. Ver si cambia la decision y defender el nuevo ADR.
